In [1]:
import shutil
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

DATA_DIR = Path("../data")
CHROMA_DIR = Path("../chroma_db")

In [2]:
shutil.rmtree(CHROMA_DIR, ignore_errors=True)
assert not CHROMA_DIR.exists(), "chroma_db is locked: close app.py, restart the kernel, run again"

In [3]:
books = pd.read_csv(DATA_DIR / "books_with_emotions.csv")
books.shape

(5197, 21)

In [4]:
documents = [
    Document(
        page_content=str(row.description).strip(), 
        metadata={"isbn13": int(row.isbn13), "title": str(row.title)},
    )
    for row in books.itertuples()
]
ids = [str(int(isbn)) for isbn in books["isbn13"]]
print(f"Created {len(documents)} documents.")

Created 5197 documents.


In [5]:
db_books = Chroma(
    collection_name="book_descriptions",
    persist_directory=str(CHROMA_DIR),
    embedding_function=OllamaEmbeddings(model="nomic-embed-text"),
    collection_metadata={"hnsw:space": "cosine"}, 
)

existing = set(db_books.get(include=[])["ids"]) 
todo = [(d, i) for d, i in zip(documents, ids) if i not in existing]

batch_size = 100
for start in range(0, len(todo), batch_size):
    batch = todo[start : start + batch_size]
    db_books.add_documents([d for d, _ in batch], ids=[i for _, i in batch])  
    print(f"Processed {min(start + batch_size, len(todo))} / {len(todo)} documents")

print("Vector database successfully built!")


Processed 100 / 5197 documents
Processed 200 / 5197 documents
Processed 300 / 5197 documents
Processed 400 / 5197 documents
Processed 500 / 5197 documents
Processed 600 / 5197 documents
Processed 700 / 5197 documents
Processed 800 / 5197 documents
Processed 900 / 5197 documents
Processed 1000 / 5197 documents
Processed 1100 / 5197 documents
Processed 1200 / 5197 documents
Processed 1300 / 5197 documents
Processed 1400 / 5197 documents
Processed 1500 / 5197 documents
Processed 1600 / 5197 documents
Processed 1700 / 5197 documents
Processed 1800 / 5197 documents
Processed 1900 / 5197 documents
Processed 2000 / 5197 documents
Processed 2100 / 5197 documents
Processed 2200 / 5197 documents
Processed 2300 / 5197 documents
Processed 2400 / 5197 documents
Processed 2500 / 5197 documents
Processed 2600 / 5197 documents
Processed 2700 / 5197 documents
Processed 2800 / 5197 documents
Processed 2900 / 5197 documents
Processed 3000 / 5197 documents
Processed 3100 / 5197 documents
Processed 3200 / 

In [6]:
TONE_TO_EMOTION = {
    "Happy": "joy",
    "Surprising": "surprise",
    "Angry": "anger",
    "Suspenseful": "fear",
    "Sad": "sadness",
}


def retrieve_semantic_recommendations(
    query, category="All", tone="All", initial_top_k=50, final_top_k=16
):
    docs = db_books.similarity_search(query, k=initial_top_k)
    ranked = [int(d.metadata["isbn13"]) for d in docs]  # CHANGED: ISBN from metadata
    rank_of = {isbn: r for r, isbn in enumerate(ranked)}

    recs = books[books["isbn13"].isin(rank_of)].copy()
    recs["similarity_rank"] = recs["isbn13"].map(rank_of)
    recs = recs.sort_values("similarity_rank")  # CHANGED: keeps similarity order

    if category != "All":  # CHANGED: new filter
        recs = recs[recs["simple_categories"] == category]
    if tone != "All":  # CHANGED: new tone re-rank
        recs = recs.sort_values(TONE_TO_EMOTION[tone], ascending=False)
    return recs.head(final_top_k).reset_index(drop=True)

In [7]:
retrieve_semantic_recommendations("A book to teach children about nature")[
    ["title", "simple_categories", "similarity_rank"]
]

,title,simple_categories,similarity_rank
0,Baby Einstein: Neighborhood Animals,Children's Fiction,0
1,Baby Einstein: Babies,Children's Fiction,1
2,The Ecological Approach to Visual Perception,Nonfiction,2
3,Baby Einstein: Birds,Children's Fiction,3
4,Baby Einstein: Dogs,Children's Fiction,4
5,The Big Box,Children's Fiction,5
6,Astronomy,Nonfiction,6
7,Judy Moody Saves the World!,Children's Fiction,7
8,The Little Big Book for God's Children,Nonfiction,8
9,Everything on a Waffle,Children's Fiction,9
